# Predict spectra with a saved LIG-PROSPECT model

This tutorial predicts four excitation wavelength/oscillator-strength pairs from new structural descriptors. It uses bundled DD (four-distance) example data; replace the paths with your own data after confirming the input conventions in `docs/input-data.md`.

Install the project first from its root: `python -m pip install -e .`

In [ ]:
from pathlib import Path

# Supports starting Jupyter in either the repository root or notebooks/.
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'pyproject.toml').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
assert (PROJECT_ROOT / 'pyproject.toml').exists(), 'Start Jupyter from the LIG-PROSPECT project directory.'
PROJECT_ROOT

## 1. Select the model and feature files

The model and input representation must agree. The bundled DD model expects four distances in the exact order used for training.

In [ ]:
MODEL_PATH = PROJECT_ROOT / 'saved_models/single_run_random_split/DD/ligprospect_dd_single_iter008_best_model.joblib'
METADATA_PATH = PROJECT_ROOT / 'saved_models/single_run_random_split/DD/ligprospect_dd_single_iter008_best_metadata.yaml'
FEATURES_DIR = PROJECT_ROOT / 'example_data/desc-4'
DESCRIPTOR = 'dd'
FILE_GLOB = '*.txt'

example_files = sorted(FEATURES_DIR.glob(FILE_GLOB))
print(f'{len(example_files)} feature files found')
print(example_files[0].name)
print(example_files[0].read_text().strip())

For your own data, replace `FEATURES_DIR` and `FILE_GLOB`. Use `*.txt` for `dd` or `add`; use `*.xyz` for `pca_cc` or `umap_ic`, while changing the model and descriptor to match.

## 2. Load the saved pipeline and prepare input features

In [ ]:
import joblib
import yaml

from lig_prospect.predict_cli import infer_expected_dim, load_features_only, predict_with_bundle

bundle = joblib.load(MODEL_PATH)
metadata = yaml.safe_load(METADATA_PATH.read_text())
expected_dim = infer_expected_dim(bundle)
print('Model expects raw feature dimension:', expected_dim)
print('Training test RMSE:', metadata.get('test_rmse'))

X_new, sample_names = load_features_only(
    descriptor=DESCRIPTOR,
    features_dir=FEATURES_DIR,
    file_glob=FILE_GLOB,
    expected_dim=expected_dim,
)
X_new.shape

## 3. Predict and save results

In [ ]:
import pandas as pd

predictions = predict_with_bundle(bundle, X_new)
columns = ['pred_1', 'osc_1', 'pred_2', 'osc_2', 'pred_3', 'osc_3', 'pred_4', 'osc_4']
results = pd.DataFrame(predictions, columns=columns)
results.insert(0, 'sample', sample_names)
results.head()

Each `pred_n` is a predicted wavelength and each matching `osc_n` is its oscillator strength. Interpret values in the units and scientific context of the training data.

In [ ]:
output_path = PROJECT_ROOT / 'outputs/prediction/notebook_example_dd_predictions.csv'
output_path.parent.mkdir(parents=True, exist_ok=True)
results.to_csv(output_path, index=False)
print(f'Saved: {output_path}')

## Next steps

- Change the model, metadata, descriptor, and input files together for another descriptor.
- Use `ligprospect-predict --config configs/predict_config.yaml` for a reproducible command-line run.
- Read `docs/input-data.md` before preparing a new mutant dataset.